# Upcycling Qwen3.5-0.8B's FFN into a trainable top-k MoE

Answers exactly one question: **at the ratio a 27B conversion actually needs,
does *training* beat the frozen ceiling?**

## Answer: no — and it gets worse the sparser the ratio

This notebook has been run: kernel `trowel3444/moeme-upcycling-poc`, one 3.25 h
session on 2x Tesla T4. Raw evidence in `results/kaggle_upcycle_poc.json`,
`results/kaggle_upcycle_poc_seed1.json` and `kaggle/kaggle_run.log`.

| E | active | frozen pilot oracle | oracle **trained** | router **trained** | beats frozen | trained ppl (dense 19.6) |
|---|---|---|---|---|---|---|
| 8 | 12.5% | 0.8624 | **0.8560** | 0.8951 | yes, by 0.006 | 109.5 |
| 32 | 3.13% | 0.9188 | **0.9355** | 0.9653 | **no** (+0.017) | 159.3 |
| 64 | 1.56% | 0.9277 | **0.9549** | 0.9806 | **no** (+0.027) | 202.5 |

Training wins only at the mildest sparsity, and **loses by more as E grows** at
the ratios a conversion needs. A second seed at E=64 reproduces the inversion to
three decimals (oracle 0.9551, router 0.9802), so it is not seed noise. The
retrofit route is closed on its own stated gate (`docs/upcycling_design.md` §9);
full write-up in `docs/experiment_log.md` E15.

Notably the *learned router* is not the fix either: it loses to the oracle at
every ratio, and the oracle route needs no router at all.

Everything below is the experiment as it was set up and run.

## What the frozen ceiling is

The local pilot (`experiments/pilot_sparsity.py`) sliced a *trained* dense FFN
into E experts, kept k of them, and updated no weights. That is a ceiling for a
*frozen* partition. For `plan=random`, k=1 on layers 0, 3, 8 and 23 it measured
0.8624 / 0.9188 / 0.9277 at E=8 / 32 / 64.

This notebook upcycles the FFN into a real MoE (`src/moeme/upcycle.py`), trains
the experts and the router on **all 24 layers**, and reports whether the trained
oracle lands below those numbers.

| metric | meaning |
|---|---|
| `val_ppl` dense / untrained / trained | the post-hoc path vs the upcycling path |
| `oracle_untrained` -> `oracle_trained` | the same measurement before and after training |
| `oracle_trained` vs `frozen_oracle` | apples to apples: both are top-k against the layer's own all-experts sum |
| `router_trained` | what the learned router actually achieves |
| `eff_experts` | routing balance: near E = healthy, near 1 = the expert set collapsed |

## What the 8-step probe established before committing the session

- The environment works: `transformers 5.6.2`, real `fineweb-edu` streaming, cuda
  with **bf16** autocast, `dense val ppl 19.6` (sane for this model on natural text).
- `causal-conv1d` **fails to build** on the free image; `flash-linear-attention`
  installs. The Gated DeltaNet therefore runs its pure-torch fallback, which is
  correct but slower -- and still fast enough: **~8.9 s per training step** at
  E=64 across all 24 layers in the probe (the first step carries a ~206 s one-off;
  the full run settled at 11.89 s/step).
- At E=64/k=1 the untrained oracle reproduced the pilot's frozen 0.9277 to within
  0.0015 on the same four layers, an independent confirmation of that measurement.
- `eff_experts` after 8 steps read 21.6 of 64; after 300 steps it read 33.2, which
  is why a probe is not a substitute for the run.

A subset of layers cannot answer the question locally: cost scales with
(upcycled layers x experts), so E=64 on all 24 layers is ~48x the 0.6 s/step the
4-layer runs cost. That is the reason this notebook exists.


In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    mem = getattr(p, "total_memory", getattr(p, "total_mem", 0))
    print(p.name, f"{mem/1e9:.1f} GB", "| bf16", torch.cuda.is_bf16_supported())


In [ ]:
# Kaggle's image ships its own CUDA-matched torch -- do not replace it.
#
# The version that actually matters is transformers. The `qwen3_5`
# architecture only exists in the 5.x line, so the floor written in
# requirements.txt (`transformers>=4.51`) can resolve to a release that cannot
# load this model at all. Pinned to the release this repo was verified against.
!pip install -q "transformers==5.6.2" "accelerate>=0.30" "safetensors>=0.4" "datasets>=2.18" pytest

# Qwen3.5's Gated DeltaNet has a fused path (causal-conv1d +
# flash-linear-attention) and a pure-torch fallback. The fallback is
# numerically correct -- verified in this repo by making both packages
# invisible to find_spec and running a real forward pass.
#
# Expectation from the probe run, so that a build failure here is not mistaken
# for a broken notebook: causal-conv1d does NOT build on the free image (no
# matching nvcc/CUDA), flash-linear-attention does install, and with only one of
# the two present the fused path is still unavailable. The run proceeds on the
# fallback and measures ~8.9 s/step at E=64. Do not spend the session chasing
# this build.
!pip install -q flash-linear-attention 2>&1 | tail -2
!pip install -q causal-conv1d 2>&1 | tail -2

import importlib.util
import transformers
print("\ntransformers", transformers.__version__)
assert importlib.util.find_spec("transformers.models.qwen3_5") is not None, (
    "This transformers has no qwen3_5 architecture, so the model cannot load. "
    "Fix the install above before running anything else."
)
for name in ("causal_conv1d", "fla"):
    print(f"  {name}: " + ("present" if importlib.util.find_spec(name)
                          else "ABSENT -> torch fallback (correct, but slower)"))
import transformers.models.qwen3_5.modeling_qwen3_5 as _q35
print("fused Gated DeltaNet path available:", _q35.is_fast_path_available,
      "(False is expected here and is not a failure)")


In [ ]:
# Get the repository onto the machine. Three routes, most reliable first:
#
#   1. set REPO_URL and clone it;
#   2. the attached Kaggle Dataset holding a tarball of the repo (what this run
#      uses). `kaggle datasets create` SKIPS directories by default, so a plain
#      folder upload arrives with no src/ at all -- the tarball is what makes the
#      contents deterministic;
#   3. a dataset that already contains the tree, found under /kaggle/input.
import glob, os, shutil, subprocess

REPO_URL = ""            # e.g. "https://github.com/<you>/moeme.git"
REPO_DIR = "/kaggle/working/moeme"

if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

if not os.path.isdir(os.path.join(REPO_DIR, "src", "moeme")):
    tarballs = sorted(set(glob.glob("/kaggle/input/*/moeme-repo.tar.gz") +
                          glob.glob("/kaggle/input/**/moeme-repo.tar.gz", recursive=True)))
    trees = glob.glob("/kaggle/input/**/src/moeme", recursive=True)
    if tarballs:
        os.makedirs(REPO_DIR, exist_ok=True)
        subprocess.run(["tar", "-xzf", tarballs[0], "-C", REPO_DIR], check=True)
        print(f"extracted {tarballs[0]} -> {REPO_DIR}")
    elif trees:
        root = os.path.dirname(os.path.dirname(trees[0]))
        shutil.copytree(root, REPO_DIR, dirs_exist_ok=True)
        print(f"copied {root} -> {REPO_DIR}")
    else:
        raise SystemExit("Attach the repo dataset, or set REPO_URL. Found no "
                         "/kaggle/input/**/moeme-repo.tar.gz and no "
                         "/kaggle/input/**/src/moeme.")

os.chdir(REPO_DIR)
assert os.path.isdir(os.path.join("src", "moeme")), "repo copy is incomplete"
assert os.path.exists(os.path.join("results", "pilot_sparsity.json")), (
    "results/pilot_sparsity.json is missing, so the frozen-oracle comparison "
    "cannot be made. Re-create the repo tarball including results/."
)
print(os.getcwd())
print(sorted(os.listdir()))


In [ ]:
# Fast plumbing check FIRST: no download, runs upcycle -> train -> residual on a tiny
# random model. Never spend a Kaggle session on an untested training loop.
!python -u experiments/upcycle_poc.py --self-test


In [ ]:
# The main run: the E trend at k=1, over all 24 layers.
#
# SIZED FROM MEASUREMENT, not from an extrapolation. The probe measured ~8.9 s/step
# at E=64 / 24 layers / seq_len 512 / grad_accum 4; the actual run came in at
# 11.89 s/step (E=64), 8.12 (E=32) and 5.05 (E=8) -- wall clocks of 3786 / 2446 /
# 1522 s, with a one-off ~206 s on the first step. `--max-seconds 5400` is a circuit
# breaker, not the expected duration: if the device is slower than the probe's, the
# run stops and still writes its report rather than dying at the session cap having
# written nothing. It did not fire on this run (`stopped_early: false` everywhere).
#
# --steps 300 matches the local E14 runs, which is what makes the two comparable.
#
# --save-every 0: a checkpoint at E=64 over 24 layers is ~1.6 GB, and five
# configurations of them would eat most of the ~20 GB output cap. Each
# configuration here is short enough that the JSON written after it is the
# better protection.
!python -u experiments/upcycle_poc.py \
    --model Qwen/Qwen3.5-0.8B \
    --experts 64 32 8 \
    --topks 1 \
    --combine sum \
    --plan random \
    --layers all \
    --steps 300 \
    --lr 1e-4 \
    --seq-len 512 \
    --grad-accum 4 \
    --corpus-chars 4000000 \
    --save-every 0 \
    --max-seconds 5400 \
    --out /kaggle/working/upcycle_poc.json


In [ ]:
# A second seed for the decisive configuration, so the E=64 result is not read as
# a seed artefact. Writes its own report; the cell below reads both.
!python -u experiments/upcycle_poc.py \
    --model Qwen/Qwen3.5-0.8B \
    --experts 64 \
    --topks 1 \
    --combine sum \
    --plan random \
    --layers all \
    --seed 1 \
    --steps 300 \
    --lr 1e-4 \
    --seq-len 512 \
    --grad-accum 4 \
    --corpus-chars 4000000 \
    --save-every 0 \
    --max-seconds 5400 \
    --out /kaggle/working/upcycle_poc_seed1.json


In [ ]:
import glob, json, os
import pandas as pd

PATHS = ["/kaggle/working/upcycle_poc.json",
         "/kaggle/working/upcycle_poc_seed1.json"]
found = [p for p in PATHS if os.path.exists(p)]
if not found:
    print("No report was written. Read the run-cell output above; its last lines "
          "say where it stopped.")
else:
    for path in found:
        rep = json.load(open(path))
        print(f"\n==== {os.path.basename(path)}")
        print(f"{rep['model']} | device={rep['device']} | amp={rep['amp_dtype']}")
        print(f"corpus: {rep.get('corpus', '?')} "
              f"({rep.get('corpus_tokens', 0):,} tokens), "
              f"{rep['train_blocks']} train / {rep['val_blocks']} val blocks "
              f"of {rep['seq_len']}")
        if rep.get("corpus") == "built-in":
            print("  ** WARNING: the requested dataset did not load, so the built-in "
                  "corpus was used. Every perplexity below is meaningless. **")
        print(f"dense val ppl {rep['dense_val_ppl']:.3f}")

        rows = []
        for r in rep["runs"]:
            c = r["config"]
            if "error" in r:
                print(f"  FAILED CONFIG E={c['n_experts']} k={c['top_k']}: {r['error']}")
                continue
            curve = r.get("loss_curve") or []
            per = None
            if len(curve) >= 2 and curve[-1]["step"] > curve[0]["step"]:
                per = ((curve[-1]["elapsed"] - curve[0]["elapsed"])
                       / (curve[-1]["step"] - curve[0]["step"]))
            rows.append({
                "E": c["n_experts"],
                "k": c["top_k"],
                "active_frac": round(c["top_k"] / c["n_experts"], 4),
                "layers": len(r["layers"]),
                "steps": f"{r['steps_completed']}/{r['steps']}"
                         + (" EARLY-STOPPED" if r["stopped_early"] else ""),
                "s_per_step": round(per, 2) if per else None,
                "dense_ppl": round(r["dense_val_ppl"], 3),
                "untrained_ppl": round(r["untrained_val_ppl"], 3),
                "trained_ppl": round(r["trained_val_ppl"], 3),
                "oracle_untrained": round(r["metrics_untrained"]["oracle_topk_mean"], 4),
                "oracle_trained": round(r["metrics_trained"]["oracle_topk_mean"], 4),
                "router_trained": round(r["metrics_trained"]["router_topk_mean"], 4),
                "eff_experts": round(r["routing"]["effective_experts_mean"], 2),
                "frozen_oracle": round(
                    r.get("frozen_oracle_mean_on_common_layers", float("nan")), 4),
                "beats_frozen": r.get("beats_frozen_oracle"),
            })
        display(pd.DataFrame(rows))
        print(f"  (effective experts are out of E; frozen_oracle is the pilot mean "
              f"on the layers it measured, 0/3/8/23)")


## How to read the table

- **`oracle_trained < frozen_oracle`** -- retraining created usable sparse structure
  that a frozen partition never had. The comparison is restricted to the layers the
  pilot measured (0, 3, 8, 23), which is why `frozen_oracle` is a mean over those
  four and not over all 24. **Result: true only at E=8** (0.8560 vs 0.8624). At E=32
  and E=64 the trained oracle is *above* the frozen one, and by more as E grows.
- **`oracle_untrained` -> `oracle_trained`** -- the same measurement on the same
  layers before and after training, so it needs no frozen reference: 0.8787->0.9118,
  0.9433->0.9721, ... at E=64. It rises. Training moves the reconstruction *away*
  from the dense output it started equal to -- `vs_original_dense` climbs at every E
  (0.9324->0.9480, 0.9839->0.9902, 0.9919->0.9974) while perplexity falls three
  orders of magnitude.
- **`oracle_trained` vs `router_trained`** -- the gap is what a better router could
  still buy. It is real (0.039 / 0.030 / 0.026 at E=8/32/64) but it is not the
  binding constraint: the *oracle* route, which needs no router, also loses to the
  frozen partition. Improving routing cannot rescue that.
- **`eff_experts`** -- out of E. Near E means the router stayed balanced. The full
  run reached 33.2 of 64 at E=64 with `load_max_over_mean` up to 9.25, so the
  default `--aux-loss-coef 1e-2` is genuinely leaving balance on the table. Note
  this is a *secondary* finding: the aux-loss sweep is worth doing, but it cannot
  explain a deficit the oracle also has.
- **`s_per_step`** -- measured, from the elapsed times stored in the report. Use it
  to size any follow-up run instead of extrapolating from another device.
- **`oracle_untrained`** should land near the pilot's frozen oracle for the same
  (E, k). It did, to within 0.0015 on the shared layers.

A caveat that applies to every perplexity here: the train and validation splits come
from the *same* 4M characters, and the trained layers saw the training half. The
trained perplexities are therefore optimistic -- the 202.5 at E=64 is a floor on the
damage, not a ceiling. The residual columns do not have this problem: they are
measured against activations captured before upcycling.

## Scaling a follow-up session

- Cost scales roughly with **(upcycled layers) x (experts)**, and the anchor is the
  measured `s_per_step` above, not a laptop extrapolation.
- `--save-every 0` here trades resumability for output space. If a later session needs
  `--resume`, re-enable checkpoints and confirm the total against the ~20 GB cap
  (~1.6 GB per checkpoint at E=64 across 24 layers).
- **Worth running next, in order:** the resident-quantisation check on the real 27B
  (`docs/decision_memo.md` Path A) -- that is where the remaining value is. Then, if
  the MoE route is revisited at all, the aux-loss sweep at E=64/k=1 and a
  `combine="softmax"` run to test the partial-sum hypothesis directly.

## What this does *not* prove

This is 0.8B with the FFN of every layer replaced. It tests the **mechanism**
-- can gradient descent make a sparse partition usable -- not a finished 27B
conversion. What it does establish is that the mechanism fails at 0.8B on this
architecture, over all 24 layers, reproducibly across seeds, at exactly the ratio the
27B design calls for. Scaling to Qwen3.8-27B needs a continued-pretraining campaign
far beyond a free tier; `docs/upcycling_design.md` is that plan and its cost.

One more honest limit: the Gated DeltaNet ran its **pure-torch fallback**, because
`causal-conv1d` does not build on the free Kaggle image. That affects speed, not
correctness -- it is the same arithmetic -- but it means the step cost measured here
is not what a fully fused kernel would give. Triton linear attention (`fla`) did
install; it is the conv1d half of the fused path that is missing.
